# Machine Learning for Credit Risk Modeling: Predicting Customer Loan Default Using Financial and Behavioral Data

### Ploblem statement

Financial institutions face significant challenges in identifying loan applicants who are likely to default while ensuring that creditworthy customers are not unfairly rejected. Traditional credit risk assessment methods may fail to capture complex relationships within large and diverse financial datasets, leading to poor lending decisions and increased financial losses. This project aims to develop a machine learning model that predicts the probability of customer loan default using the Home Credit Default Risk dataset. By leveraging customer demographic information, financial history, credit bureau records, previous loan applications, installment payments, credit card activity, and POS/Cash loan history, the model seeks to improve credit risk assessment and support more accurate, data-driven lending decisions.

## Importing Libraries

In [29]:
#  Core data handling 
import os
import gc
import itertools
import random
import numpy as np
import pandas as pd

#  Data source 
from datasets import load_dataset

# Visualization 
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing 
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Model selection / splitting 
from sklearn.model_selection import train_test_split, RandomizedSearchCV

#  Feature selection 
from sklearn.feature_selection import mutual_info_classif, RFE

#  Models 
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
import xgboost as xgb
import lightgbm as lgb
import catboost as cb

#  Evaluation metrics 
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_score,
    recall_score, f1_score, confusion_matrix, classification_report,accuracy_score, roc_curve
)

# Interpretability 
import shap

#  Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

### Downloading the raw data

Pulls all 8 tables from the Hugging Face mirror of the competition. Saves each as a CSV into the `data/` folder.

In [30]:

# Make sure the data folder exists

folder = "data"
os.makedirs(folder, exist_ok=True)

# Home Credit dataset configurations

configs = {
    "application_train_dated": "application_train.csv",
    "application_test": "application_test.csv",
    "bureau": "bureau.csv",
    "bureau_balance": "bureau_balance.csv",
    "previous_application": "previous_application.csv",
    "POS_CASH_balance": "POS_CASH_balance.csv",
    "credit_card_balance": "credit_card_balance.csv",
    "installments_payments": "installments_payments.csv",
    "sample_submission": "sample_submission.csv"
}

print("=" * 60)
print("Downloading Home Credit Default Risk Dataset")
print("=" * 60)

for config_name, output_file in configs.items():

    print(f"\nLoading: {config_name}")

    try:
        dataset = load_dataset(
            "mohameddhameem/home-credit-default-risk",
            config_name
        )

         # Convert the HF dataset split to a pandas DataFrame
        df = dataset["train"].to_pandas()

         # Save to CSV so the rest of the notebook can work with plain pandas
        save_path = os.path.join("data", output_file)
        df.to_csv(save_path, index=False)

        print(f"✓ Saved: {output_file}")
        print(f"  Shape: {df.shape}")

        # Free memory before moving to the next (large) table
        del dataset
        del df
        gc.collect()

    except Exception as e:
        print(f"✗ Failed: {config_name}")
        print(e)

print("\n" + "=" * 60)
print("Download Complete!")
print("=" * 60)


Loading: application_train_dated
✓ Saved: application_train.csv
  Shape: (307511, 123)

Loading: application_test
✓ Saved: application_test.csv
  Shape: (48744, 121)

Loading: bureau
✓ Saved: bureau.csv
  Shape: (1716428, 17)

Loading: bureau_balance
✓ Saved: bureau_balance.csv
  Shape: (27299925, 3)

Loading: previous_application
✓ Saved: previous_application.csv
  Shape: (1670214, 37)

Loading: POS_CASH_balance
✓ Saved: POS_CASH_balance.csv
  Shape: (10001358, 8)

Loading: credit_card_balance
✓ Saved: credit_card_balance.csv
  Shape: (3840312, 23)

Loading: installments_payments
✓ Saved: installments_payments.csv
  Shape: (13605401, 8)

Loading: sample_submission
✓ Saved: sample_submission.csv
  Shape: (48744, 2)

Download Complete!


Due to account verification issues with Kaggle, I was unable to access the dataset directly through the Kaggle API. As an alternative, I downloaded the Home Credit Default Risk dataset from the Hugging Face repository, which mirrors the original Kaggle competition dataset. This ensured I could proceed with the project while using the same data structure and features as the original competition dataset.

In [31]:
# Load the primary applicant-level table and take a first look
train = pd.read_csv("data/application_train.csv")

print("Shape:", train.shape)
train.head()

Shape: (307511, 123)


,SK_ID_CURR,application_date,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,2018-01-01,1,Cash loans,M,N,Y,0,202500.0,406597.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,2018-01-01,0,Cash loans,F,N,N,0,270000.0,1293502.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,2018-01-01,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,2018-01-01,0,Cash loans,F,N,Y,0,135000.0,312682.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,2018-01-01,0,Cash loans,M,N,Y,0,121500.0,513000.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


Loaded my training dataset into Python to give me a quick overview of what it contains.

Do a data validation and integrity check. Before I begin the analysis, I need to  verify that all the required dataset files are present, appear complete, and contain the expected columns.

In [32]:
folder = "data"

expected_files = [
    "application_train.csv", "application_test.csv", "bureau.csv",
    "bureau_balance.csv", "previous_application.csv", "POS_CASH_balance.csv",
    "credit_card_balance.csv", "installments_payments.csv"
]

missing = [f for f in expected_files if not os.path.exists(os.path.join(folder, f))]
if missing:
    print(f"✗ MISSING: {missing}")
else:
    print(f"All {len(expected_files)} expected files found in '{folder}'")

check = pd.read_csv(os.path.join(folder, "application_train.csv"), nrows=5)
print("TARGET column present:", "TARGET" in check.columns)

All 8 expected files found in 'data'
TARGET column present: True


In [33]:

files = [
    "application_train.csv",
    "application_test.csv",
    "bureau.csv",
    "bureau_balance.csv",
    "previous_application.csv",
    "POS_CASH_balance.csv",
    "credit_card_balance.csv",
    "installments_payments.csv"
]

for file in files:
    df = pd.read_csv(f"data/{file}")
    print(f"{file}: {len(df):,} rows")

application_train.csv: 307,511 rows
application_test.csv: 48,744 rows
bureau.csv: 1,716,428 rows
bureau_balance.csv: 27,299,925 rows
previous_application.csv: 1,670,214 rows
POS_CASH_balance.csv: 10,001,358 rows
credit_card_balance.csv: 3,840,312 rows
installments_payments.csv: 13,605,401 rows


Before I start working with the data, I run two quick checks.

First, I make sure all 8 files are actually there, and that the main file has the TARGET column - that's the column that tells me who defaulted and who didn't, so if it's missing, nothing else in the project can work.

Second, I open each file and count how many rows it has. This helps me see how big each table is before I start combining them. For example, bureau_balance has over 27 million rows - way more than one row per person - which is why I had to shrink it down to one row per applicant before I could use it.

Basically, these two checks make sure everything is in place before I start the real work, so if something's wrong, I catch it early instead of getting a confusing error later on."

TARGET is a column with one value per applicant, and it only has two possible values:

0 = did not default → they repaid the loan

1 = did default → they failed to repay

##  Feature Engineering: Aggregate the Secondary Tables


There are 5 extra tables, and each one has many rows per person (like many past loans per applicant). Since our main table only has one row per person, we will first squish each of these 5 tables down to one row per person - using things like the average, total, and worst-case value to summarize their history. Then we merged each of these summarized tables onto our main table, matching rows by the person's ID. After each merge, we will check that the number of rows don't change, just to make sure nothing gets duplicated by mistake. 

In [34]:

# Define the folder where the input datasets are stored
INPUT_DIR = './data'

# Define the folder where processed files or outputs will be saved
OUTPUT_DIR = './output'

# Create the output folder if it does not already exist
# exist_ok=True prevents an error if the folder already exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

def downcast(df, exclude=('SK_ID_CURR',)):
    """Shrink dtypes to reduce memory footprint (float64->float32, int64->smallest int, etc)."""

    # Loop through every column in the dataframe one at a time
    for col in df.columns:

        if col in exclude:       # Skip columns listed in the exclude parameter
            continue

        if df[col].dtype == 'float64':             # Convert float64 columns to float32 This cuts memory usage roughly in half while maintaining good precision
            df[col] = df[col].astype('float32')

        elif df[col].dtype == 'int64':     # Convert int64 columns to the smallest possible integer type (e.g., int8, int16, int32) depending on the values stored
            df[col] = pd.to_numeric(df[col], downcast='integer')

        # If it's True/False, store it as a tiny integer (0 or 1) instead
        elif df[col].dtype == 'bool':
            df[col] = df[col].astype('int8')

        # If it's text, store it as a "category" type instead of plain text    
        elif df[col].dtype == object:
            df[col] = df[col].astype('category')
            
    return df

The Home Credit dataset contains millions of records across several CSV files. Therfore, two folders are created,  one to read the raw data from, and one to save our results into.This makes sure that save folder actually exists, to avoid crashing later when we try to write a file to it.


The **downcast function** is about saving memory. When data is loaded into into pandas, it often uses more computer memory than it needs to. For example, it might use a big number type to store something small, like someone's age. That's a waste of space.

So this function goes through every column and checks what kind of data it holds, then switches it to a smaller, lighter version that still holds the same information correctly:

* Decimal numbers - switched to a smaller decimal type
* Whole numbers - switched to the smallest whole-number type that fits
* True/False - switched to just 0 or 1
* Text - switched to a more compact text format 

By optimizing data types:

- Memory usage is significantly reduced.
- Data loads and operations become faster.
- The risk of running out of RAM is lower.
- Machine learning preprocessing becomes more efficient, especially when merging multiple large tables.

#### A. Aggregate **bureau_balance** - one row per **SK_ID_BUREAU**

This table is monthly loan status history at *other* credit institutions, keyed by ***SK_ID_BUREAU*** (not ***SK_ID_CURR***), so it has to be aggregated up one level before it can attach to ***bureau***.

In [35]:
# Read the CSV file using optimized data types to reduce memory usage.
# bureau_balance is a largest table in the project (~27 million rows), so specifying smaller data types saves a significant amount of RAM.
bb = pd.read_csv(
    f"{INPUT_DIR}/bureau_balance.csv",
    dtype={"SK_ID_BUREAU": "int32", "MONTHS_BALANCE": "int16", "STATUS": "category"},   # Tell pandas exactly what size/type to use for each column while loading,
    # instead of letting it guess, this saves a lot of memory right from the start
)
print("Loaded bureau_balance:", bb.shape)

# Convert each repayment status into separate binary columns.
# One-hot encode the monthly status flag (0-5 = days-past-due bucket, C = closed, X = unknown)
# This turns the single STATUS column into several 0/1 columns, one per status value
status_dummies = pd.get_dummies(bb["STATUS"], prefix="BB_STATUS")
# Keep only the required columns and combine them with the new dummy variables
bb = pd.concat([bb[["SK_ID_BUREAU", "MONTHS_BALANCE"]], status_dummies], axis=1)

# Aggregate to one row per SK_ID_BUREAU: how many months of history, its span, and how many
# months fell into each status bucket

# Aggregate Monthly Records
# Each loan (SK_ID_BUREAU) has many monthly records. summarize them into a single row
bb_agg = bb.groupby("SK_ID_BUREAU").agg(
    BB_MONTHS_COUNT=("MONTHS_BALANCE", "count"),        # Total number of months recorded
    BB_MONTHS_MIN=("MONTHS_BALANCE", "min"),            # Earliest month available
    BB_MONTHS_MAX=("MONTHS_BALANCE", "max"),            # Most recent month available
)

# Sum each dummy column.
# Since the dummy columns contain only 0 and 1, summing gives the number of months in each status.
bb_agg = bb_agg.join(bb.groupby("SK_ID_BUREAU")[status_dummies.columns].sum())

# Convert the grouped index (loan ID) back into a normal column
bb_agg.reset_index(inplace=True)

# Display the aggregated dataset
print("Aggregated bureau_balance shape:", bb_agg.shape)
# Store the summarized dataset as a Parquet file. Parquet files are smaller, faster to read, and more efficient than CSV files.
bb_agg.to_parquet(f"{OUTPUT_DIR}/agg_bureau_balance.parquet", index=False)

# Delete large objects that are no longer needed. ( The original dataset is very large and is no longer needed after aggregation. Deleting it frees up memory for the next processing steps.)
del bb, status_dummies
# Force Python's garbage collector to release memory.
gc.collect()

Loaded bureau_balance: (27299925, 3)
Aggregated bureau_balance shape: (817395, 12)


0

This table (bureau_balance) is huge - about 27 million rows - and it tracks the monthly status of loans that people had at other banks, not Home Credit itself. Each row is one month of one loan's history.

Since it's so big, the code loads it carefully, telling pandas the exact (small) data type for each column up front, instead of loading it the normal way and shrinking it after.

Each loan's monthly status is a code like "0" (up to date), "1-5" (increasingly overdue), "C" (closed), or "X" (unknown). The code turns that one status column into several yes/no columns  one per possible status  so it can be counted and summed later.

Then it groups everything by loan ID and narrows each loan down to one summary row: how many months of history it has, the first and last month, and how many months it spent in each status.

Finally, it saves this smaller summary table to disk, and clears the big original table out of memory since it's no longer needed - this keeps the notebook from running out of memory when working with such a large file.

#### B. Aggregate **bureau** (+ bureau_balance features) - one row per **SK_ID_CURR**

Past loans at other credit institutions. Each applicant can have many rows here (many past loans), so we one-hot the categoricals, fold in the ***bureau_balance*** aggregates, then group by ***SK_ID_CURR*** with mean/sum/min/max.

In [36]:
# Load the bureau dataset

# Read the bureau.csv file into a DataFrame.
# This dataset contains information about each applicant's previous loans from external credit bureaus.bureau = pd.read_csv(f"{INPUT_DIR}/bureau.csv")
bureau = pd.read_csv(f"{INPUT_DIR}/bureau.csv")

print("Loaded bureau:", bureau.shape)

# Merge with the aggregated bureau_balance dataset. Combine bureau.csv with the aggregated bureau_balance
# using SK_ID_BUREAU as the common key.
# how="left" ensures that every bureau loan is kept, even if it has no matching bureau_balance record.

bureau = bureau.merge(bb_agg, on="SK_ID_BUREAU", how="left")

# One-hot encode the categorical columns (Convert each category into separate binary columns.)
# This allows machine learning algorithms to use categorical variables.
for col in ["CREDIT_ACTIVE", "CREDIT_CURRENCY", "CREDIT_TYPE"]:
    bureau = pd.concat([bureau, pd.get_dummies(bureau[col], prefix=col)], axis=1)

# Select Numeric Columns for Aggregation
# Numeric columns get mean/sum/min/max per applicant (captures typical + extreme loan behavior)
num_cols = [
    # Days since the credit was granted
    "DAYS_CREDIT",
    # Number of overdue days
    "CREDIT_DAY_OVERDUE",
    # Expected credit end date
    "DAYS_CREDIT_ENDDATE",
    # Actual credit closing date
    "DAYS_ENDDATE_FACT",
    # Maximum overdue credit amount
    "AMT_CREDIT_MAX_OVERDUE",
    # Number of times the loan was prolonged
    "CNT_CREDIT_PROLONG",
    # Total credit amount
    "AMT_CREDIT_SUM",
    # Outstanding debt
    "AMT_CREDIT_SUM_DEBT",
    # Credit limit
    "AMT_CREDIT_SUM_LIMIT",
    # Amount currently overdue
    "AMT_CREDIT_SUM_OVERDUE",
    # Last bureau update
    "DAYS_CREDIT_UPDATE",
    # Loan annuity
    "AMT_ANNUITY",
    # Aggregated bureau_balance variables
    "BB_MONTHS_COUNT",
    "BB_MONTHS_MIN",
    "BB_MONTHS_MAX",
]
# Find the names of the new  columns we just created above, to summarize them too
bb_status_cols = [c for c in bureau.columns if c.startswith("BB_STATUS_")]
dummy_cols = [c for c in bureau.columns if c.startswith(("CREDIT_ACTIVE_", "CREDIT_CURRENCY_", "CREDIT_TYPE_"))]

# Define Aggregation Rules
# For every numeric column calculate:Mean, Sum, Minimum, Maximum
agg_dict = {c: ["mean", "sum", "min", "max"] for c in num_cols + bb_status_cols}
# For dummy variables calculate only the mean. Example:CREDIT_ACTIVE_Active Mean = proportion of loans that are Active.
agg_dict.update({c: "mean" for c in dummy_cols})     
# Count the total number of bureau loans each applicant has. 
agg_dict["SK_ID_BUREAU"] = "count" 

# Aggregate to One Row Per Customer
# Group all rows by person (applicant ID), and apply the summary plan above
bureau_agg = bureau.groupby("SK_ID_CURR").agg(agg_dict)
# Rename the resulting columns to be clear and consistent
bureau_agg.columns = ["BUREAU_" + "_".join(col).upper() for col in bureau_agg.columns]
# Turn the applicant ID from an index back into a normal column
bureau_agg.reset_index(inplace=True)

print("Aggregated bureau shape:", bureau_agg.shape)
bureau_agg.to_parquet(f"{OUTPUT_DIR}/agg_bureau.parquet", index=False)

del bureau, bb_agg
gc.collect()


Loaded bureau: (1716428, 17)
Aggregated bureau shape: (305811, 117)


0

This picks up the bureau table - every past loan a person had at other banks, not Home Credit. One person can appear many times here (once per past loan), so just like before, everything needs to be squished down to one row per person.

First, it attaches the monthly-history summary from the earlier step, matching by loan ID, so each loan now also carries info like how many months of history it had.

Then it converts the text columns (like loan status: active, closed, etc.) into simple yes/no columns, since summary math like "average" or "sum" only works on numbers, not text.

After that, it groups everything by person and boils each person's loans down into one row, calculating for each number column: the average, the total, the smallest, and the largest value. This captures both the "typical" loan behavior and the worst-case one (like their biggest overdue amount). For the yes/no columns, it just takes the average, which tells you what share of a person's loans fall into each category (for example, what percent were closed loans).

It also counts how many past loans each person had in total.

Finally, it renames the columns clearly, saves the result, and clears the big tables out of memory to keep things running smoothly.

#### C. Aggregate **POS_CASH_balance** - one row per **SK_ID_CURR**

Monthly balance history on POS/cash loans. This table already carries ***SK_ID_CURR*** directly, so no intermediate ***SK_ID_PREV*** step is needed.

In [37]:
# Load the POS_CASH_balance dataset

# Read the dataset while specifying smaller data types to reduce memory usage.
pos = pd.read_csv(
    f"{INPUT_DIR}/POS_CASH_balance.csv",
    dtype={
        "SK_ID_PREV": "int32",                 # Previous loan ID
        "SK_ID_CURR": "int32",                 # Customer ID
        "MONTHS_BALANCE": "int16",             # Months before current application
        "CNT_INSTALMENT": "float32",           # Total scheduled installments
        "CNT_INSTALMENT_FUTURE": "float32",    # Remaining installments
        "NAME_CONTRACT_STATUS": "category",    # Loan status
        "SK_DPD": "int32",                     # Days Past Due
        "SK_DPD_DEF": "int32",                 # Days Past Due with tolerance
    },
)
print("Loaded POS_CASH_balance:", pos.shape)

# One-hot encode contract status (Active, Completed, Signed, etc.)
status_dummies = pd.get_dummies(pos["NAME_CONTRACT_STATUS"], prefix="POS_STATUS")
# Remove the original status column and append the newly created dummy columns.
pos = pd.concat([pos.drop(columns=["NAME_CONTRACT_STATUS"]), status_dummies], axis=1)

# Define Aggregation Rules

# For numerical variables calculate:Mean,Maximum
agg_dict = {c: ["mean", "max"] for c in ["CNT_INSTALMENT", "CNT_INSTALMENT_FUTURE", "SK_DPD", "SK_DPD_DEF"]}

# For dummy variables calculate the mean. The mean represents the proportion of loans in each contract status.
agg_dict.update({c: "mean" for c in status_dummies.columns})

# Count the number of unique previous POS/Cash loans belonging to each customer.
agg_dict["SK_ID_PREV"] = "nunique" 

# Aggregate to One Row Per Customer
# Group all POS/Cash loans belonging to each customer.
pos_agg = pos.groupby("SK_ID_CURR").agg(agg_dict)
# Rename Columns
pos_agg.columns = ["POS_" + "_".join(col).upper() for col in pos_agg.columns]

# Convert SK_ID_CURR back into a normal column.
pos_agg.reset_index(inplace=True)

print("Aggregated POS_CASH_balance shape:", pos_agg.shape)
pos_agg.to_parquet(f"{OUTPUT_DIR}/agg_pos_cash.parquet", index=False)

del pos, status_dummies
gc.collect()

Loaded POS_CASH_balance: (10001358, 8)
Aggregated POS_CASH_balance shape: (337252, 19)


0

The POS_CASH_balance.csv table records the month-by-month repayment history of customers' Point-of-Sale (POS) and cash loans, meaning each customer can appear many times-once for each month of each loan. To make the data suitable for machine learning, the code summarizes these records into one row per customer (SK_ID_CURR).

It first converts the loan status (e.g., Active, Completed) into numeric dummy variables, then groups all records belonging to the same customer. For each customer, it calculates key features such as the average and maximum number of installments, the average and maximum days past due, the proportion of loans in each contract status, and the total number of unique POS/Cash loans. Finally, it renames the columns, saves the aggregated dataset, and frees memory by deleting the large raw dataset, ensuring efficient processing of the remaining tables.

#### D. Aggregate **credit_card_balance** -  one row per **SK_ID_CURR**

In [38]:

# Load the Credit Card Balance Dataset


# Read the credit_card_balance.csv file.
# This dataset contains monthly information about customers'credit card accounts.

cc = pd.read_csv(f"{INPUT_DIR}/credit_card_balance.csv")

# Display the number of rows and columns loaded
print("Loaded credit_card_balance:", cc.shape)


# One-Hot Encode the Contract Status

# Convert each contract status (e.g., Active, Completed) into binary (0/1) columns.

status_dummies = pd.get_dummies(
    cc["NAME_CONTRACT_STATUS"],
    prefix="CC_STATUS"
)

# Remove the original contract status column and append the dummy variables.

cc = pd.concat(
    [
        cc.drop(columns=["NAME_CONTRACT_STATUS"]),
        status_dummies
    ],
    axis=1
)

# Select Numeric Columns
# These are the numerical variables that describe customers' credit card usage and repayment behavior.

num_cols = [
    # Current outstanding balance
    "AMT_BALANCE",
    # Credit limit
    "AMT_CREDIT_LIMIT_ACTUAL",
    # ATM cash withdrawals
    "AMT_DRAWINGS_ATM_CURRENT",
    # Total withdrawals
    "AMT_DRAWINGS_CURRENT",
    # Other withdrawals
    "AMT_DRAWINGS_OTHER_CURRENT",
    # POS purchases
    "AMT_DRAWINGS_POS_CURRENT",
    # Minimum payment due
    "AMT_INST_MIN_REGULARITY",
    # Current payment
    "AMT_PAYMENT_CURRENT",
    # Total payment
    "AMT_PAYMENT_TOTAL_CURRENT",
    # Principal receivable
    "AMT_RECEIVABLE_PRINCIPAL",
    # Receivable amount
    "AMT_RECIVABLE",
    # Total receivable
    "AMT_TOTAL_RECEIVABLE",
    # Number of ATM withdrawals
    "CNT_DRAWINGS_ATM_CURRENT",
    # Total number of withdrawals
    "CNT_DRAWINGS_CURRENT",
    # Other withdrawals count
    "CNT_DRAWINGS_OTHER_CURRENT",
    # POS withdrawal count
    "CNT_DRAWINGS_POS_CURRENT",
    # Number of completed installments
    "CNT_INSTALMENT_MATURE_CUM",
    # Days past due
    "SK_DPD",
    # Days past due with tolerance
    "SK_DPD_DEF"
]

# Define Aggregation Rules

# For each numeric variable calculate:Mean ,Maximum, Sum

agg_dict = {
    c: ["mean", "max", "sum"]
    for c in num_cols
}

# For contract status dummy variables,calculate the mean (proportion).
agg_dict.update({c: "mean"for c in status_dummies.columns})

# Count the number of unique credit card accounts each customer has had.

agg_dict["SK_ID_PREV"] = "nunique"

# Aggregate to One Row Per Customer
# Group all credit card records belonging to each customer.

cc_agg = cc.groupby("SK_ID_CURR").agg(agg_dict)

# Rename Columns
# Create descriptive column names.
cc_agg.columns = ["CC_" + "_".join(col).upper()for col in cc_agg.columns]

# Convert SK_ID_CURR back into a normal column.
cc_agg.reset_index(inplace=True)

# Display Results
print("Aggregated credit_card_balance shape:", cc_agg.shape)

# Save the Aggregated Dataset
cc_agg.to_parquet(f"{OUTPUT_DIR}/agg_credit_card.parquet",index=False)

# Free Memory
# Delete objects that are no longer needed.
del cc, status_dummies
# Force Python to release unused memory.
gc.collect()

Loaded credit_card_balance: (3840312, 23)
Aggregated credit_card_balance shape: (103558, 66)


0

The credit_card_balance.csv table contains the month-by-month history of customers' credit card accounts, so the same customer can appear many times-once for each month of each credit card account. To prepare the data for machine learning, the code summarizes these records into one row per customer (SK_ID_CURR).

It first converts the credit card contract status (such as Active or Completed) into numeric dummy variables, then groups all records belonging to the same customer. For each customer, it calculates the average, maximum, and total values for key credit card features such as balances, credit limits, payments, withdrawals, receivables, and days past due. It also calculates the proportion of records in each contract status and counts the number of unique credit card accounts each customer has had. Finally, it renames the columns with clear prefixes, saves the aggregated dataset, and frees memory by deleting the large raw dataset before processing the next table.

#### E. Aggregate **previous_application** - one row per **SK_ID_CURR**


In [39]:
# Load the Previous Application Dataset

# Read the previous_application.csv file. Each row represents one previous loan application made by a customer.
prev = pd.read_csv(f"{INPUT_DIR}/previous_application.csv")
print("Loaded previous_application:", prev.shape)

# In this dataset, the value 365243 is a placeholdermeaning "date not available".
# Replace it with NaN (missing value) so it is ignored during statistical calculations.
for col in ["DAYS_FIRST_DRAWING", "DAYS_FIRST_DUE", "DAYS_LAST_DUE_1ST_VERSION", "DAYS_LAST_DUE", "DAYS_TERMINATION"]:
    prev[col] = prev[col].replace(365243, np.nan)

# One-hot encode all categorical columns, then take the per-applicant mean of each dummy
# (= "share of previous applications that had this category")
cat_cols = [
    "NAME_CONTRACT_TYPE", "FLAG_LAST_APPL_PER_CONTRACT", "NAME_CASH_LOAN_PURPOSE",
    "NAME_CONTRACT_STATUS", "NAME_PAYMENT_TYPE", "CODE_REJECT_REASON", "NAME_TYPE_SUITE",
    "NAME_CLIENT_TYPE", "NAME_GOODS_CATEGORY", "NAME_PORTFOLIO", "NAME_PRODUCT_TYPE",
    "CHANNEL_TYPE", "NAME_SELLER_INDUSTRY", "NAME_YIELD_GROUP", "PRODUCT_COMBINATION",
]
# Start with the customer ID.
dummy_frames = [prev[["SK_ID_CURR"]]]
# Store the names of all dummy columns.
dummy_cols = []
# Convert each categorical column into dummy variables.
for col in cat_cols:
    d = pd.get_dummies(prev[col], prefix=col)
    dummy_cols.extend(d.columns.tolist())
    dummy_frames.append(d)
# Combine all dummy variables into one DataFrame.
prev_dummies = pd.concat(dummy_frames, axis=1)

# Numeric columns get mean/max/min/sum
num_cols = [
    "AMT_ANNUITY", "AMT_APPLICATION", "AMT_CREDIT", "AMT_DOWN_PAYMENT", "AMT_GOODS_PRICE",
    "RATE_DOWN_PAYMENT", "RATE_INTEREST_PRIMARY", "RATE_INTEREST_PRIVILEGED", "DAYS_DECISION",
    "SELLERPLACE_AREA", "CNT_PAYMENT", "DAYS_FIRST_DRAWING", "DAYS_FIRST_DUE",
    "DAYS_LAST_DUE_1ST_VERSION", "DAYS_LAST_DUE", "DAYS_TERMINATION", "NFLAG_INSURED_ON_APPROVAL",
]

# For every numeric column calculate:Mean, Maximum, Minimum, Sum
agg_dict = {c: ["mean", "max", "min", "sum"] for c in num_cols}
# Group all previous applications belonging to each customer.
prev_num_agg = prev.groupby("SK_ID_CURR").agg(agg_dict)
prev_num_agg.columns = ["PREV_" + "_".join(col).upper() for col in prev_num_agg.columns]

# Calculate the mean of every dummy column.
# This represents the proportion of previous applications in each category.
prev_dummy_agg = prev_dummies.groupby("SK_ID_CURR")[dummy_cols].mean()
# Rename columns.
prev_dummy_agg.columns = ["PREV_" + c.upper() + "_MEAN" for c in prev_dummy_agg.columns]
# Count how many previous applications each customer has submitted.
prev_count = prev.groupby("SK_ID_CURR")["SK_ID_PREV"].count().rename("PREV_COUNT")
# Merge All Aggregated Features
# Combine: Numeric summaries, Dummy summaries, Application count
prev_agg = prev_num_agg.join(prev_dummy_agg).join(prev_count)
# Convert SK_ID_CURR back into a normal column.
prev_agg.reset_index(inplace=True)
#  Display Results
print("Aggregated previous_application shape:", prev_agg.shape)
# Save the Aggregated Dataset
prev_agg.to_parquet(f"{OUTPUT_DIR}/agg_previous_app.parquet", index=False)

# Free Memory

# Delete large DataFrames that are no longer needed.

del prev
del prev_dummies
del prev_num_agg
del prev_dummy_agg

# Release unused memory.

gc.collect()

Loaded previous_application: (1670214, 37)
Aggregated previous_application shape: (338857, 206)


0

The previous_application.csv table contains information about every previous loan application that customers submitted to Home Credit (as opposed to the bureau table, which contains loans from other financial institutions). Since one customer may have submitted multiple applications over time, they can appear many times in this table. To prepare the data for machine learning, the code summarizes all previous applications into one row per customer (SK_ID_CURR).

The first step addresses a data quality issue where several date columns use the value 365243 as a placeholder to indicate that no date is available. These placeholder values are replaced with NaN (missing values) so they are ignored during statistical calculations and do not distort measures such as averages, minimums, or maximums.

Next, all categorical (text) variables, such as loan purpose, contract status, payment type, and rejection reason, are converted into numeric dummy variables, allowing them to be included in the aggregation process.

The customer-level summary is then created in three parts. First, for each numerical feature-such as loan amounts, interest rates, installment information, and application dates-the code calculates the mean, maximum, minimum, and total across all previous applications. Second, for each dummy variable, it calculates the mean, which represents the proportion of a customer's previous applications that belong to each category (for example, the share of applications that were approved or refused). Finally, it counts the total number of previous loan applications submitted by each customer.

The three sets of aggregated features are then combined into a single customer-level dataset. Lastly, the columns are renamed with clear PREV_ prefixes, the summarized dataset is saved as a Parquet file, and the large intermediate DataFrames are removed from memory to improve efficiency before processing the next dataset.

#### F. Aggregate **installments_payments**

In [40]:

# Load the installments_payments table - the actual record of whether each loan installment was paid on time and in full
inst = pd.read_csv(f"{INPUT_DIR}/installments_payments.csv")
print("Loaded installments_payments:", inst.shape)

# Create New Payment Behaviour Features

# Calculate how many days late (or early) a payment was made.
# Positive value  -> payment was late.
# Zero            -> payment was made on time.
# Negative value  -> payment was made early.
inst["DAYS_LATE"] = inst["DAYS_ENTRY_PAYMENT"] - inst["DAYS_INSTALMENT"]       
# Calculate the difference between the expected installment amount and the amount actually paid.
# Positive value -> customer underpaid.
# Zero           -> exact payment.
# Negative value -> customer paid more than required.
inst["AMT_PAYMENT_DIFF"] = inst["AMT_INSTALMENT"] - inst["AMT_PAYMENT"]      

# Calculate the payment ratio.
# 1.0  -> paid exactly the required amount.
# >1   -> overpaid.
# <1   -> underpaid.
# Replace zero installment values with NaN to avoid division-by-zero errors.   
inst["AMT_PAYMENT_RATIO"] = inst["AMT_PAYMENT"] / inst["AMT_INSTALMENT"].replace(0, np.nan)

# List of number columns we want to summarize for each person
num_cols = [
    "NUM_INSTALMENT_VERSION", "DAYS_INSTALMENT", "DAYS_ENTRY_PAYMENT",
    "AMT_INSTALMENT", "AMT_PAYMENT", "DAYS_LATE", "AMT_PAYMENT_DIFF", "AMT_PAYMENT_RATIO",
]

# Define Aggregation Rules

# For every numeric feature calculate:, Mean, Maximum, Minimum, Sum, Standard deviation
agg_dict = {c: ["mean", "max", "min", "sum", "std"] for c in num_cols}
# Count the number of unique previous loans with installment records.
agg_dict["SK_ID_PREV"] = "nunique"       
# Count the total number of installment records.        
agg_dict["NUM_INSTALMENT_NUMBER"] = "count"      

# Aggregate to One Row Per Customer
# Group all installment records belongingto each customer.
inst_agg = inst.groupby("SK_ID_CURR").agg(agg_dict)
# Rename Columns

# Create descriptive column names beginning with INSTAL_.
inst_agg.columns = ["INSTAL_" + "_".join(col).upper() for col in inst_agg.columns]
# Convert SK_ID_CURR back into a normal column.
inst_agg.reset_index(inplace=True)

# Display Results
print("Aggregated installments_payments shape:", inst_agg.shape)
# Save the Aggregated Dataset
inst_agg.to_parquet(f"{OUTPUT_DIR}/agg_installments.parquet", index=False)


# Delete the original large DataFrame.
del inst
# Release unused memory.
gc.collect()

Loaded installments_payments: (13605401, 8)
Aggregated installments_payments shape: (339587, 43)


0

The installments_payments.csv table contains the detailed repayment history for customers' previous loans, recording every installment payment made. Since each customer can have multiple loans and each loan consists of many installments, a customer may appear many times in this table. To prepare the data for machine learning, the code summarizes all installment records into **one row per customer (SK_ID_CURR)**.

Before performing the aggregation, the code creates three new features that capture payment behaviour. It calculates **DAYS_LATE**, which measures how many days early or late a payment was (with positive values indicating late payments), **AMT_PAYMENT_DIFF**, which measures the difference between the expected installment amount and the amount actually paid (where positive values indicate underpayment), and **AMT_PAYMENT_RATIO**, which shows the proportion of the expected installment that was paid. These engineered features provide valuable insights into a customer's repayment behaviour and are often strong indicators of future default risk.

The data is then grouped by customer, and for each numerical feature the code calculates the **mean, maximum, minimum, sum, and standard deviation**. These statistics capture not only the customer's typical payment behaviour but also their highest and lowest values, total payment activity, and the consistency of their repayment patterns. The code also counts the **number of unique previous loans** with installment records and the **total number of installments** associated with each customer.

Finally, the aggregated features are renamed using clear *INSTAL_* prefixes, the summarized dataset is saved as a Parquet file, and the large raw dataset is removed from memory to improve efficiency before processing the next table.


#### G. Merge everything onto ***application_train*** / ***application_test***

Each secondary table is now one row per `SK_ID_CURR`, so a left-merge won't duplicate any rows. We verify row counts stay constant after every merge as a sanity check.

In [41]:
 # Load the main application file (either train or test)
def build_features(app_path, is_train):
    df = pd.read_csv(app_path)
    # Reduce memory usage by converting columns to smaller data types
    df = downcast(df)
    # Store the original number of rows.
    # This will later be used to verify that merging does not accidentally duplicate or remove customers.
    start_rows = df.shape[0]

# List of Aggregated Feature Tables
   # These datasets were created during the previous preprocessing steps.
    pieces = [
        f'{OUTPUT_DIR}/agg_bureau.parquet',
        f'{OUTPUT_DIR}/agg_pos_cash.parquet',
        f'{OUTPUT_DIR}/agg_credit_card.parquet',
        f'{OUTPUT_DIR}/agg_previous_app.parquet',
        f'{OUTPUT_DIR}/agg_installments.parquet',
    ]

  # Merge Every Aggregated Dataset
 # Loop through each summarized table one at a time and attach it
    for path in pieces:
        piece = downcast(pd.read_parquet(path))       # Optimize memory usage
        # Attach it onto the main table, matching rows by applicant ID
        # "left" means: keep every row in df, even if a person has no matching data in this table (those columns just become blank)
        df = df.merge(piece, on='SK_ID_CURR', how='left')
         # Verify that the number of customers
        # has not changed after merging.
        assert df.shape[0] == start_rows, f"Row count changed after merging {path}!"

       # Delete the temporary dataset
        del piece
        # Release unused memory
        gc.collect()
    # Display Final Dataset Shape
    print(f"{'train' if is_train else 'test'} final shape:", df.shape)
       # Return the Final Dataset
    return df
# Build the Training Dataset
train = build_features(f'{INPUT_DIR}/application_train.csv', is_train=True)
# Build the Testing Dataset
test = build_features(f'{INPUT_DIR}/application_test.csv', is_train=False)

train final shape: (307511, 569)
test final shape: (48744, 567)


This function takes the main application file and glues all 5 of the summarized tables from earlier onto it, one at a time, matching them up by applicant ID.

It starts by loading the application data and shrinking its memory size, then notes down how many people (rows) are in it before anything else happens.

Then it loops through the 5 summary files one by one, and for each one:

- Loads it and shrinks it too
- Attaches it onto the main table using a "left" merge - this means every original applicant stays in the table no matter what, and if they don't have any history in that particular table (say, they never had a credit card), those new columns just come out blank for them
- Immediately checks that the number of rows hasn't changed. Since each summary table has exactly one row per applicant, a proper merge should never create extra rows. If it did, that would mean something went wrong earlier (like a duplicate ID slipping through), so the code stops right away with an error message instead of silently continuing with bad data
- Clears that summary table from memory before moving to the next one, to keep things running smoothly

Once all 5 tables are attached, it prints the final size and returns the combined table.

At the bottom, this same function is called twice - once to build the full feature table for train, and once for test - so both datasets go through exactly the same process and end up with the same set of columns.

#### 8. Align train/test columns and save

One-hot encoding can occasionally produce a category in train that never appears in test (or vice versa). Reindexing test to train's exact column set (filling any gaps with 0) prevents this from silently breaking prediction later.

Therefore, I need to have the training and testing datasets ready for machine learning for the model expects both datasets to have the same features, the code makes sure they match before saving them.

In [42]:

# Remove Unnecessary Columns

# Some datasets may contain columns that are not useful for model training. Remove them if they exist.

for c in ['application_date']:
    if c in train.columns:
        train = train.drop(columns=[c])            # Remove from the training dataset
    if c in test.columns:
        test = test.drop(columns=[c])               # Remove from the testing dataset


# Create the List of Feature Columns

# Get the list of all feature columns — everything except the answer column
# (TARGET) and the ID column (SK_ID_CURR), since those aren't model inputs
feature_cols = [c for c in train.columns if c not in ('TARGET', 'SK_ID_CURR')]
# Reorder the test dataset so its columns match the training dataset exactly.
# Keep SK_ID_CURR first because it is needed later to identify customers.
# If any feature is missing in the test set, create it and fill it with 0.
test = test.reindex(columns=['SK_ID_CURR'] + feature_cols, fill_value=0)


# Verify the Columns Match

# Confirm that the training and testing datasets contain exactly the same feature columns.
# TARGET exists only in the training dataset,so it is excluded from the comparison.
assert list(train.columns.drop('TARGET')) == list(test.columns), "Column mismatch between train/test!"

# Display Dataset Shapes
print("train:", train.shape, "| test:", test.shape)
print("Columns aligned OK.")

# Save the processed datasets as Parquet files.
train.to_parquet(f'{OUTPUT_DIR}/train_final.parquet', index=False)
test.to_parquet(f'{OUTPUT_DIR}/test_final.parquet', index=False)
print("Saved train_final.parquet and test_final.parquet")

train: (307511, 568) | test: (48744, 567)
Columns aligned OK.
Saved train_final.parquet and test_final.parquet


The Code above first removes any columns that should not be used for training, such as the raw application_date column. Next, it creates a list of all the feature columns by excluding the customer ID (SK_ID_CURR) and the target variable (TARGET), since these are not input features for the model.

The code then reshapes the test dataset so it has the same columns and the same column order as the training dataset. If the test dataset is missing any feature columns, they are added and filled with 0.

An assert statement is used to check that the training and testing datasets now have matching columns. If they do not, the code immediately stops with an error, preventing problems during model training.

Finally, both datasets are saved as Parquet files. This allows the processed data to be loaded quickly in future without repeating the entire preprocessing pipeline.

In [43]:
print(train.shape)
train.head()

(307511, 568)


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,INSTAL_AMT_PAYMENT_DIFF_MIN,INSTAL_AMT_PAYMENT_DIFF_SUM,INSTAL_AMT_PAYMENT_DIFF_STD,INSTAL_AMT_PAYMENT_RATIO_MEAN,INSTAL_AMT_PAYMENT_RATIO_MAX,INSTAL_AMT_PAYMENT_RATIO_MIN,INSTAL_AMT_PAYMENT_RATIO_SUM,INSTAL_AMT_PAYMENT_RATIO_STD,INSTAL_SK_ID_PREV_NUNIQUE,INSTAL_NUM_INSTALMENT_NUMBER_COUNT
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0.0,0.000000,0.000000,1.000000,1.0,1.00000,19.0,0.000000,1.0,19.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0.0,0.000000,0.000000,1.000000,1.0,1.00000,25.0,0.000000,3.0,25.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0.0,0.000000,0.000000,1.000000,1.0,1.00000,3.0,0.000000,1.0,3.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0.0,0.000000,0.000000,1.000000,1.0,1.00000,16.0,0.000000,3.0,16.0
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0.0,29857.365234,2843.383545,0.954545,1.0,0.00005,63.0,0.209751,5.0,66.0


In [44]:
print(test.shape)
test.head()

(48744, 567)


,SK_ID_CURR,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,...,INSTAL_AMT_PAYMENT_DIFF_MIN,INSTAL_AMT_PAYMENT_DIFF_SUM,INSTAL_AMT_PAYMENT_DIFF_STD,INSTAL_AMT_PAYMENT_RATIO_MEAN,INSTAL_AMT_PAYMENT_RATIO_MAX,INSTAL_AMT_PAYMENT_RATIO_MIN,INSTAL_AMT_PAYMENT_RATIO_SUM,INSTAL_AMT_PAYMENT_RATIO_STD,INSTAL_SK_ID_PREV_NUNIQUE,INSTAL_NUM_INSTALMENT_NUMBER_COUNT
0,100001,Cash loans,F,N,Y,0,135000.0,568800.0,20560.5,450000.0,...,0.0,0.000000,0.000000,1.000000,1.0,1.000000,7.0,0.000000,2.0,7.0
1,100005,Cash loans,M,N,Y,0,99000.0,222768.0,17370.0,180000.0,...,0.0,0.000000,0.000000,1.000000,1.0,1.000000,9.0,0.000000,1.0,9.0
2,100013,Cash loans,M,Y,Y,0,202500.0,663264.0,69777.0,630000.0,...,0.0,179437.718750,4844.349609,0.935484,1.0,0.000266,145.0,0.241331,4.0,155.0
3,100028,Cash loans,F,N,Y,2,315000.0,1575000.0,49018.5,1575000.0,...,0.0,70348.226562,1724.086914,0.911504,1.0,0.030496,103.0,0.213056,3.0,113.0
4,100038,Cash loans,M,Y,N,1,180000.0,625500.0,32067.0,625500.0,...,0.0,0.000000,0.000000,1.000000,1.0,1.000000,12.0,0.000000,1.0,12.0


In [45]:
train.to_csv(f'{OUTPUT_DIR}/train_final.csv', index=False)
test.to_csv(f'{OUTPUT_DIR}/test_final.csv', index=False)
print("Saved CSVs to", OUTPUT_DIR)

Saved CSVs to ./output
